In [1]:
from cobra.io import read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite
from cobra.io.sbml import validate_sbml_model
import cobra
import pandas as pd
import numpy as np
from glob import glob

In [2]:
cobra_config = cobra.Configuration()
cobra_config

Attribute,Description,Value
solver,Mathematical optimization solver,gurobi
tolerance,"General solver tolerance (feasibility, integrality, etc.)",1e-07
lower_bound,Default reaction lower bound,-1000.0
upper_bound,Default reaction upper bound,1000.0
processes,Number of parallel processes,7
cache_directory,Path for the model cache,/home/acs98/.cache/cobrapy
max_cache_size,Maximum cache size in bytes,104857600
cache_expiration,Model cache expiration time in seconds (if any),None


In [3]:
b_cel = read_sbml_model("../vmh/Bacteroides_cellulosilyticus_DSM_14838.xml")
b_cel

Set parameter Username
Academic license - for non-commercial use only - expires 2025-01-11


Name,Bacteroides_cellulosilyticus_DSM_14838
Memory address,7f2e181f3280
Number of metabolites,1513
Number of reactions,2278
Number of genes,877
Number of groups,77
Objective expression,1.0*biomass536 - 1.0*biomass536_reverse_50d4e
Compartments,"Cytoplasm, Extracellular"


In [4]:
# ovatus_vmh = read_sbml_model("../vmh/Bacteroides_ovatus_ATCC_8483.xml")
# ovatus_vmh

In [5]:
# ovatus_vmh.metabolites.get_by_id('pect[e]')

In [6]:
# ovatus_vmh.metabolites.get_by_id('xylan[e]')

In [7]:
# ovatus_vmh.metabolites.get_by_id('arabinogal[e]')

In [8]:
# # Get the metabolite from ovatus_vmh model
# metabolite = ovatus_vmh.metabolites.get_by_id('pect[e]')
# # Add the metabolite to b_cel model
# b_cel.add_metabolites(metabolite)
# # Get the reactions associated with the metabolite from ovatus_vmh model
# reactions = ovatus_vmh.reactions.query(lambda x: metabolite in x.metabolites)
# b_cel.add_reactions(reactions)

In [9]:
# # Get the metabolite from ovatus_vmh model
# metabolite = ovatus_vmh.metabolites.get_by_id('xylan[e]')
# # Add the metabolite to b_cel model
# b_cel.add_metabolites(metabolite)
# # Get the reactions associated with the metabolite from ovatus_vmh model
# reactions = ovatus_vmh.reactions.query(lambda x: metabolite in x.metabolites)
# b_cel.add_reactions(reactions)

In [10]:
b_cel.id = 'Bacteroides_cellulosilyticus_WH2'
b_cel.name = 'Bacteroides cellulosilyticus WH2'
b_cel

Name,Bacteroides_cellulosilyticus_WH2
Memory address,7f2e181f3280
Number of metabolites,1513
Number of reactions,2278
Number of genes,877
Number of groups,77
Objective expression,1.0*biomass536 - 1.0*biomass536_reverse_50d4e
Compartments,"Cytoplasm, Extracellular"


In [11]:
models = [read_sbml_model(f) for f in glob("../vmh/Bacteroides_*.xml")]
models.append(read_sbml_model("../vmh/Parabacteroides_distasonis_ATCC_8503.xml"))
models.pop(2)
models

[<Model Bacteroides_uniformis_ATCC_8492 at 0x7f2dc13153d0>,
 <Model Bacteroides_thetaiotaomicron_VPI_5482 at 0x7f2dbf210eb0>,
 <Model Bacteroides_caccae_ATCC_43185 at 0x7f2dbde8aa90>,
 <Model Bacteroides_ovatus_ATCC_8483 at 0x7f2dbc248e20>,
 <Model Bacteroides_cellulosilyticus_DSM_14838 at 0x7f2dbb0daf40>,
 <Model Parabacteroides_distasonis_ATCC_8503 at 0x7f2dbbfa4c10>]

In [12]:
models_rxns = [pd.DataFrame([r.id for r in i.reactions], columns=[i.name]) for i in models]
models_rxns

[     Bacteroides uniformis ATCC 8492
 0                            23DHMPO
 1                             23PDE2
 2                             23PDE4
 3                             23PDE7
 4                             23PDE9
 ...                              ...
 2413                           r2137
 2414                  rtranscription
 2415                     sink_dmbzid
 2416                          sink_s
 2417                      biomass536
 
 [2418 rows x 1 columns],
      Bacteroides thetaiotaomicron VPI-5482
 0                                   12PPDt
 1                                  23DHMPO
 2                                   23PDE2
 3                                   23PDE4
 4                                   23PDE7
 ...                                    ...
 2519                                 r2137
 2520                        rtranscription
 2521                         sink_gthrd(c)
 2522                                sink_s
 2523                           

In [13]:
# Get all model reactions names in a dataframe
b_cel_rxns = pd.DataFrame([r.id for r in b_cel.reactions], columns=[b_cel.name])
b_cel_rxns

,Bacteroides cellulosilyticus WH2
0,12PPDt
1,23DHMPO
2,23PDE2
3,23PDE4
4,23PDE7
...,...
2273,r2137
2274,rtranscription
2275,sink_dmbzid
2276,sink_s


In [14]:
print(f'{b_cel_rxns.columns[0]}: {b_cel_rxns.shape[0]}')
for i in models_rxns:
    print(f'{i.columns[0]}: {i.shape[0]}')

Bacteroides cellulosilyticus WH2: 2278
Bacteroides uniformis ATCC 8492: 2418
Bacteroides thetaiotaomicron VPI-5482: 2524
Bacteroides caccae ATCC 43185: 2381
Bacteroides ovatus ATCC 8483: 2476
Bacteroides cellulosilyticus DSM 14838: 2278
Parabacteroides distasonis ATCC 8503: 2519


In [15]:
rxns_to_add = []
for model in models_rxns:
    rxns_to_add.append(model[~model[model.columns[0]].isin(b_cel_rxns[b_cel_rxns.columns[0]])])
rxns_to_add

[     Bacteroides uniformis ATCC 8492
 96                           ABUTt2r
 113                         ACNAMtNa
 114                            ACNML
 132                           ACPpds
 154                           ADPRDP
 ...                              ...
 2378                           URIK5
 2379                           URIK6
 2380                           URIK7
 2381                           URIK8
 2390                      XYLAN_DEGe
 
 [298 rows x 1 columns],
      Bacteroides thetaiotaomicron VPI-5482
 97                                  7AHSDH
 98                                 7AHSDHe
 100                                 AACOAT
 119                               ACNAMabc
 151                                 ADMDCr
 ...                                    ...
 2479                                 URAt2
 2493                                 XANt2
 2494                                  XPPT
 2521                         sink_gthrd(c)
 2523                            

In [16]:
rxns_to_add_merged = rxns_to_add[0].copy()
for i in range(1, len(rxns_to_add)):
    rxns_to_add_merged = rxns_to_add_merged.merge(rxns_to_add[i], how='outer', left_on=rxns_to_add_merged.columns[0], right_on=rxns_to_add[i].columns[0])
rxns_to_add_merged

,Bacteroides uniformis ATCC 8492,Bacteroides thetaiotaomicron VPI-5482,Bacteroides caccae ATCC 43185,Bacteroides ovatus ATCC 8483,Bacteroides cellulosilyticus DSM 14838,Parabacteroides distasonis ATCC 8503
0,ABUTt2r,NaN,ABUTt2r,ABUTt2r,NaN,NaN
1,ACNAMtNa,NaN,NaN,NaN,NaN,NaN
2,ACNML,NaN,ACNML,ACNML,NaN,ACNML
3,ACPpds,NaN,NaN,NaN,NaN,NaN
4,ADPRDP,NaN,ADPRDP,ADPRDP,NaN,NaN
...,...,...,...,...,...,...
751,NaN,NaN,NaN,NaN,NaN,TREabc
752,NaN,NaN,NaN,NaN,NaN,XANt2
753,NaN,NaN,NaN,NaN,NaN,XPPT
754,NaN,NaN,NaN,NaN,NaN,sink_gthrd(c)


In [17]:
rxns_to_add_unique = pd.unique(rxns_to_add_merged.values.flatten())
rxns_to_add_unique

array(['ABUTt2r', nan, 'ACNAMtNa', 'ACNML', 'ACPpds', 'ADPRDP',
       'AHCYSNS_r', 'AMPSO3OX', 'APAT', 'ARBt2r', 'ARGt2r', 'ASADi',
       'CDPGLYCGPT', 'CGLYabc', 'CMPSAS', 'CPPPGO2', 'CYSR', 'CYSTL',
       'CYSt2r', 'DAPabc', 'DCYSDS', 'DM_dhptd(c)', 'EX_26dap_M(e)',
       'EX_4abut(e)', 'EX_C02528(e)', 'EX_MGlcn54(e)', 'EX_MGlcn54_rl(e)',
       'EX_acnam(e)', 'EX_cgly(e)', 'EX_cholate(e)', 'EX_cit(e)',
       'EX_cys_L(e)', 'EX_dgchol(e)', 'EX_dsT_antigen(e)',
       'EX_dsT_antigen_rl(e)', 'EX_galt(e)', 'EX_gchola(e)', 'EX_gly(e)',
       'EX_h2s(e)', 'EX_kesto(e)', 'EX_kestottr(e)', 'EX_lac_L(e)',
       'EX_lald_L(e)', 'EX_melib(e)', 'EX_pac(e)', 'EX_phe_L(e)',
       'EX_taur(e)', 'EX_tchola(e)', 'EX_tdchola(e)', 'EX_tre(e)',
       'EX_trp_L(e)', 'EX_xylan(e)', 'FCLTc', 'FEDCabc', 'FMNRx2',
       'GALTpts', 'GALUi', 'GALt2_2', 'GCDCHOLBHSe', 'GCHOLBHSe',
       'GLYt2r', 'H2St', 'HSERTA', 'INS2D', 'IOR2', 'IOR3', 'KESTOASEe',
       'KESTOTTRASEe', 'LCADi', 'LLALDt', 'LYSt

In [18]:
rxns_to_add_unique_by_species = {}
for model in models:
    rxns_to_add_unique_by_species[model.name] = [model.reactions.get_by_id(i) for i in rxns_to_add_unique if i in model.reactions]
rxns_to_add_unique_by_species

{'Bacteroides uniformis ATCC 8492': [<Reaction ABUTt2r at 0x7f2dd2af4fd0>,
  <Reaction ACNAMtNa at 0x7f2dbfd68fd0>,
  <Reaction ACNML at 0x7f2dbfd68c70>,
  <Reaction ACPpds at 0x7f2dbfd676a0>,
  <Reaction ADPRDP at 0x7f2dbfd612e0>,
  <Reaction AHCYSNS_r at 0x7f2dbfd1ce80>,
  <Reaction AMPSO3OX at 0x7f2dbfd24e50>,
  <Reaction APAT at 0x7f2dbfd28fd0>,
  <Reaction ARBt2r at 0x7f2dbfd29760>,
  <Reaction ARGt2r at 0x7f2dbfd2aa30>,
  <Reaction ASADi at 0x7f2dbfd29a00>,
  <Reaction CDPGLYCGPT at 0x7f2dbfbbfaf0>,
  <Reaction CGLYabc at 0x7f2dbfbb39a0>,
  <Reaction CMPSAS at 0x7f2dbfb44e20>,
  <Reaction CPPPGO2 at 0x7f2dbfb5bd00>,
  <Reaction CYSR at 0x7f2dbfb67be0>,
  <Reaction CYSTL at 0x7f2dbfb54fd0>,
  <Reaction CYSt2r at 0x7f2dbfb67ee0>,
  <Reaction DAPabc at 0x7f2dbfb14820>,
  <Reaction DCYSDS at 0x7f2dbfb2a6d0>,
  <Reaction DM_dhptd(c) at 0x7f2dbfad1eb0>,
  <Reaction EX_26dap_M(e) at 0x7f2dbfa994c0>,
  <Reaction EX_4abut(e) at 0x7f2dbfa8ee50>,
  <Reaction EX_C02528(e) at 0x7f2dbfa8edc0>,

In [19]:
rxns_to_add_unique_by_species['Bacteroides ovatus ATCC 8483']

[<Reaction ABUTt2r at 0x7f2dbc0e0340>,
 <Reaction ACNML at 0x7f2dbc088a00>,
 <Reaction ADPRDP at 0x7f2dbc0929a0>,
 <Reaction AHCYSNS_r at 0x7f2dbc04beb0>,
 <Reaction AMPSO3OX at 0x7f2dbc053af0>,
 <Reaction ARGt2r at 0x7f2dbc057e80>,
 <Reaction CPPPGO2 at 0x7f2dbbc2ff40>,
 <Reaction CYSTL at 0x7f2dbbbc1f10>,
 <Reaction CYSt2r at 0x7f2dbbc36eb0>,
 <Reaction EX_4abut(e) at 0x7f2dbbb64520>,
 <Reaction EX_C02528(e) at 0x7f2dbbb64ac0>,
 <Reaction EX_MGlcn54(e) at 0x7f2dbba99460>,
 <Reaction EX_MGlcn54_rl(e) at 0x7f2dbba99580>,
 <Reaction EX_acnam(e) at 0x7f2dbba58e20>,
 <Reaction EX_cholate(e) at 0x7f2dbba6b820>,
 <Reaction EX_cys_L(e) at 0x7f2dbba73d60>,
 <Reaction EX_dgchol(e) at 0x7f2dbba73820>,
 <Reaction EX_dsT_antigen(e) at 0x7f2dbba7e460>,
 <Reaction EX_dsT_antigen_rl(e) at 0x7f2dbba7e520>,
 <Reaction EX_galt(e) at 0x7f2dbba7efd0>,
 <Reaction EX_gchola(e) at 0x7f2dbba066a0>,
 <Reaction EX_gly(e) at 0x7f2dbba069a0>,
 <Reaction EX_h2s(e) at 0x7f2dbba06d00>,
 <Reaction EX_kesto(e) at 0x7

In [20]:
# Add reactions to b_cel model
b_cel.add_reactions(rxns_to_add_unique_by_species['Bacteroides ovatus ATCC 8483'])

In [21]:
ov = rxns_to_add_merged.loc[:,['Bacteroides ovatus ATCC 8483']].copy().dropna().values[:,0]
pa = rxns_to_add_merged.loc[:,['Parabacteroides distasonis ATCC 8503']].copy().dropna().values[:,0]
set_diff = list(set(pa) - set(ov))
set_diff

['DM_clpni17(c)',
 'PLIPA2E161',
 'LPLIPAL2A141',
 'CDPGLYCGPT',
 'EX_q8(e)',
 'AMID4',
 'ITDUDPMG',
 'PGPGT',
 'PLIPA2E181',
 'AMANK',
 'CAt4i',
 'TECA2S180',
 'SUDPDGT',
 'LPLIPAL2E120',
 'PLIPA1G181',
 'HETT',
 'DM_5MTR',
 'LPLIPAL1G120',
 'PLIPA1E181',
 'TECA1SAI15',
 'LPLIPAL2A161',
 'GLYCLTt2r',
 'SBTD_D2',
 'TECA2S160',
 'O16G2e',
 'LPLIPAL1E120',
 'TECA2SI16',
 'TAURabc',
 'PLIPA2E120',
 'TECA1SI14',
 'PTRCtex2',
 'ARBt2r',
 'PUTA3',
 'DEXTRINabc',
 'TREabc',
 'MUDPMGT',
 'TECA2SI17',
 'EX_fum(e)',
 'GGTT',
 'OMBZLM',
 'PLIPA2G181',
 'AMID2',
 'PLIPA1A120',
 'LPLIPAL1G160',
 'EX_glyclt(e)',
 'PLIPA1G180',
 'PLIPA2A120',
 'LPLIPAL1E181',
 'PLIPA2G140',
 'LPLIPAL2G160',
 'TECA2SI14',
 'GLTPD',
 'OIVD2',
 'CGLYt3_2_',
 'SHCHF',
 'IHDUDPMG',
 'IND3ACt2r',
 'LPLIPAL2E181',
 'IXDUDPMG',
 'ISOMALT',
 'PLIPA1E140',
 'PLIPA2G120',
 'GluForTx',
 'LPLIPAL2G141',
 'MLTG3',
 'PLIPA1A180',
 'PLIPA2A160',
 'PLIPA2E180',
 'LPLIPAL2G161',
 'ADMDCr',
 'LPLIPAL1A180',
 'NO3abc',
 'HYPD',
 'GCALDD

In [22]:
annot = pd.read_table('recon-store-reactions-1.tsv')
pa_annot = annot[annot['abbreviation'].isin(set_diff)]
pa_annot

,abbreviation,description,formula,subsystem,ecnumber,keggId,keggorthology,cog,seed,metanetx
7,AMY1e,"Alpha-Amylase, Extracellular (Strch1 -> Strch2)",8.0 h2o[e] + strch1[e] -> 8.0 glc_D[e] + strch...,Starch and sucrose metabolism,NaN,NaN,K01176,COG0366,NaN,MNXR95832
9,CGLYt3_2_,Cys-Gly Transport in via Proton Symport,cgly[e] + 2.0 h[e] -> cgly[c] + 2.0 h[c],"Transport, extracellular",NaN,NaN,NaN,NaN,NaN,NaN
24,EX_strch1(e),"Exchange of Starch, Structure 1 (1, 6-{7[1, 4-...",strch1[e] <=>,Exchange/demand reaction,NaN,NaN,NaN,NaN,NaN,NaN
25,EX_strch2(e),"Exchange of Starch, Structure 2 (1, 6-{2[1, 4-...",strch2[e] <=>,Exchange/demand reaction,NaN,NaN,NaN,NaN,NaN,NaN
32,GluForTx,Glutamate Formimidoyltransferase,forglu[c] + h[c] + thf[c] -> 5forthf[c] + glu_...,Histidine metabolism,"2.1.2.5, 4.3.1.4",R02287,NaN,NaN,NaN,MNXR100281
...,...,...,...,...,...,...,...,...,...,...
1387,biomass345,Biomass0,0.0030965 10fthf[c] + 0.0030965 2dmmq8[c] + 0....,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2333,PTRCtex2,putrescine excretion (cytosol to extracellular),ptrc[c] -> ptrc[e],"Transport, extracellular",NaN,NaN,NaN,NaN,NaN,NaN
2338,FTCD,formimidoyltransferase cyclodeaminase,5forthf[c] + 2.0 h[c] -> methf[c] + nh4[c],Folate metabolism,NaN,NaN,NaN,NaN,NaN,NaN
2517,4HPRO_LTt,trans-4-hydroxy-L-proline transport,4hpro_LT[e] + h[e] -> 4hpro_LT[c] + h[c],"Transport, extracellular",NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
pa_annot['subsystem'].unique()

array(['Starch and sucrose metabolism', 'Transport, extracellular',
       'Exchange/demand reaction', 'Histidine metabolism',
       'Purine catabolism', 'Phenylalanine metabolism',
       'Tetrahydrobiopterin metabolism',
       'Fructose and mannose metabolism', 'Thiamine metabolism',
       'Aminosugar metabolism',
       'Ubiquinone and other terpenoid-quinone biosynthesis',
       'Valine, leucine, and isoleucine metabolism',
       'Vitamin B12 metabolism', 'Cell wall biosynthesis',
       'Glycerophospholipid metabolism',
       'Arginine and Proline Metabolism', 'Aminobenzoate degradation',
       'Terpenoid backbone biosynthesis', 'Glutamate metabolism',
       'Cholesterol metabolism', 'N-glycan synthesis', 'Urea cycle',
       'Inositol phosphate metabolism', 'Pyruvate metabolism',
       'Galactose metabolism', 'Energy metabolism',
       'Nucleotide interconversion', 'Propanoate metabolism',
       'Glyoxylate and dicarboxylate metabolism',
       'Arginine and proline me

In [24]:
subsy_int = ['Starch and sucrose metabolism', 
             'Fructose and mannose metabolism',
             'Aminosugar metabolism',
             'Glycerophospholipid metabolism',
             'N-glycan synthesis',
             'Pyruvate metabolism',
             'Galactose metabolism',
             'Glyoxylate and dicarboxylate metabolism'
             ]

In [25]:
pa_annot_int = pa_annot[pa_annot['subsystem'].isin(subsy_int)]
pa_annot_int

,abbreviation,description,formula,subsystem,ecnumber,keggId,keggorthology,cog,seed,metanetx
7,AMY1e,"Alpha-Amylase, Extracellular (Strch1 -> Strch2)",8.0 h2o[e] + strch1[e] -> 8.0 glc_D[e] + strch...,Starch and sucrose metabolism,NaN,NaN,K01176,COG0366,NaN,MNXR95832
46,O16G2e,"Oligo-1, 6-Glucosidase (Strch2 ->Strch3), Extr...",h2o[e] + strch2[e] -> glc_D[e] + malt[e],Starch and sucrose metabolism,NaN,NaN,K01182,COG0366,NaN,MNXR102084
54,SBTD_D2,D-Sorbitol Dehydrogenase (D-Fructose Producing),nad[c] + sbt_D[c] -> fru[c] + h[c] + nadh[c],Fructose and mannose metabolism,"1.1.1.14, 1.1.1.15",R00875,K00008,COG1063,rxn00634,MNXR104283
91,AMANAPEr,N-acetylmannosamine 6-phosphate epimerase,acmanap[c] <=> acgam6p[c],Aminosugar metabolism,5.1.3.9,NaN,K01788 K13967,COG3010,rxn01506,MNXR81674
229,LPLIPAL1A120,Lysophospholipase L1 (2-acylglycerophosphotida...,1ddecg3p[c] + h2o[c] -> ddca[c] + glyc3p[c] + ...,Glycerophospholipid metabolism,3.1.1.5,NaN,K01048,COG2267,rxn08796,MNXR3269
...,...,...,...,...,...,...,...,...,...,...
1245,GCALDDr,Glycolaldehyde dehydrogenase,gcald[c] + h2o[c] + nad[c] <=> glyclt[c] + 2.0...,Glyoxylate and dicarboxylate metabolism,1.2.1.21,NaN,K07248,COG1012,rxn00979,MNXR2565
1255,AIHDUDPMG,Anteisoheptadecanoyl-UDP-glucosyltransferase (...,12daihdglyc[c] + udpg[c] <=> h[c] + m12dahglyc...,Glycerophospholipid metabolism,NaN,NaN,NaN,NaN,rxn10284,MNXR80529
1256,AIHDUPDG,Anteisoheptadecanoyl-UDP-glucosyltransferase (...,m12dahglyc[c] + udpg[c] <=> diglcpgai17[c] + h...,Glycerophospholipid metabolism,NaN,NaN,NaN,NaN,rxn10275,MNXR80520
1257,AIPDUDPG,Anteisopentadecanoyl-UDP-glucosyltransferase (...,m12dapglyc[c] + udpg[c] <=> diglcpgai15[c] + h...,Glycerophospholipid metabolism,NaN,NaN,NaN,NaN,rxn10278,MNXR80523


In [26]:
pa_annot_int_id = pa_annot_int['abbreviation'].values
pa_annot_int_id

array(['AMY1e', 'O16G2e', 'SBTD_D2', 'AMANAPEr', 'LPLIPAL1A120',
       'LPLIPAL1A140', 'LPLIPAL1A141', 'LPLIPAL1A160', 'LPLIPAL1A161',
       'LPLIPAL1A180', 'LPLIPAL1A181', 'LPLIPAL1E120', 'LPLIPAL1E140',
       'LPLIPAL1E141', 'LPLIPAL1E160', 'LPLIPAL1E161', 'LPLIPAL1E180',
       'LPLIPAL1E181', 'LPLIPAL1G120', 'LPLIPAL1G140', 'LPLIPAL1G141',
       'LPLIPAL1G160', 'LPLIPAL1G161', 'LPLIPAL1G180', 'LPLIPAL1G181',
       'LPLIPAL2A120', 'LPLIPAL2A140', 'LPLIPAL2A141', 'LPLIPAL2A160',
       'LPLIPAL2A161', 'LPLIPAL2A180', 'LPLIPAL2A181', 'LPLIPAL2E120',
       'LPLIPAL2E140', 'LPLIPAL2E141', 'LPLIPAL2E160', 'LPLIPAL2E161',
       'LPLIPAL2E180', 'LPLIPAL2E181', 'LPLIPAL2G120', 'LPLIPAL2G140',
       'LPLIPAL2G141', 'LPLIPAL2G160', 'LPLIPAL2G161', 'LPLIPAL2G180',
       'LPLIPAL2G181', 'PLIPA1A120', 'PLIPA1A140', 'PLIPA1A141',
       'PLIPA1A160', 'PLIPA1A161', 'PLIPA1A180', 'PLIPA1A181',
       'PLIPA1E120', 'PLIPA1E140', 'PLIPA1E141', 'PLIPA1E160',
       'PLIPA1E161', 'PLIPA1E180re

In [27]:
# Get pa_interest_reactions
pa_model = read_sbml_model("../vmh/Parabacteroides_distasonis_ATCC_8503.xml")

In [28]:
pa_reactions = pa_model.reactions.query(lambda x: x.id in pa_annot_int_id)
pa_reactions

[<Reaction ACALD at 0x7f2db9350b20>,
 <Reaction AIHDUDPMG at 0x7f2db929ddf0>,
 <Reaction AIHDUPDG at 0x7f2db929de20>,
 <Reaction AIPDUDPG at 0x7f2db92a5b80>,
 <Reaction AIPDUDPMG at 0x7f2db92a5d30>,
 <Reaction AMANAPEr at 0x7f2db92bdfd0>,
 <Reaction AMANK at 0x7f2db92bdfa0>,
 <Reaction AMY1e at 0x7f2db924a9d0>,
 <Reaction APAT at 0x7f2db924f850>,
 <Reaction DEXTRINASE at 0x7f2db8f80fd0>,
 <Reaction FT at 0x7f2db8de4670>,
 <Reaction G3PAT180 at 0x7f2db8deceb0>,
 <Reaction G3PATAI17 at 0x7f2db8decb80>,
 <Reaction G3PATI17 at 0x7f2db8ce9a90>,
 <Reaction GCALDDr at 0x7f2db8c9a250>,
 <Reaction GLTPD at 0x7f2db8c40d30>,
 <Reaction GLYK at 0x7f2db8c7ba90>,
 <Reaction GMAND at 0x7f2db8c11370>,
 <Reaction ISOMALT at 0x7f2db8b9d970>,
 <Reaction LPLIPAL1A120 at 0x7f2db8b6cbe0>,
 <Reaction LPLIPAL1A140 at 0x7f2db8b6cfa0>,
 <Reaction LPLIPAL1A141 at 0x7f2db8b6cfd0>,
 <Reaction LPLIPAL1A160 at 0x7f2db8b6cc40>,
 <Reaction LPLIPAL1A161 at 0x7f2db8b79640>,
 <Reaction LPLIPAL1A180 at 0x7f2db8b79820>,
 <

In [29]:
# Get the metabolites in the reactions
pa_metabolites = []
for i in pa_reactions:
    pa_metabolites.extend(i.metabolites.keys())
pa_metabolites = list(set(pa_metabolites))
pa_metabolites

[<Metabolite ocdcea[c] at 0x7f2db93f80a0>,
 <Metabolite 1agpg140[c] at 0x7f2db93da0d0>,
 <Metabolite pa161[c] at 0x7f2db93f80d0>,
 <Metabolite ttdca[c] at 0x7f2db9380100>,
 <Metabolite 1agpg141[c] at 0x7f2db93da100>,
 <Metabolite 1agpg120[c] at 0x7f2db93da130>,
 <Metabolite pa120[c] at 0x7f2db93f8160>,
 <Metabolite 1agpg160[c] at 0x7f2db93da1c0>,
 <Metabolite pa160[c] at 0x7f2db93f81c0>,
 <Metabolite m12dahglyc[c] at 0x7f2db93f61f0>,
 <Metabolite 1agpg161[c] at 0x7f2db93da1f0>,
 <Metabolite 1agpg180[c] at 0x7f2db93da220>,
 <Metabolite 1agpg181[c] at 0x7f2db93da250>,
 <Metabolite nad[c] at 0x7f2db93f8280>,
 <Metabolite 1aihpdecg3p[c] at 0x7f2db93da280>,
 <Metabolite nadh[c] at 0x7f2db93f82b0>,
 <Metabolite 1ddecg3p[c] at 0x7f2db93da2e0>,
 <Metabolite 1hdec9eg3p[c] at 0x7f2db93da310>,
 <Metabolite 1hdecg3p[c] at 0x7f2db93da340>,
 <Metabolite 1ihpdecg3p[c] at 0x7f2db93da3a0>,
 <Metabolite g3pe[c] at 0x7f2db93f23a0>,
 <Metabolite ttc_ggdp[c] at 0x7f2db93803d0>,
 <Metabolite g3pg[c] at 0x7f

In [30]:
b_cel.add_metabolites(pa_metabolites)

In [31]:
for i in pa_metabolites:
    rxns = pa_model.reactions.query(lambda x: i in x.metabolites)
    b_cel.add_reactions(rxns)

Ignoring reaction 'FACOAL181' since it already exists.
Ignoring reaction 'AGPAT161' since it already exists.
Ignoring reaction 'DAGK161' since it already exists.
Ignoring reaction 'DASYN161' since it already exists.
Ignoring reaction 'FACOAL140' since it already exists.
Ignoring reaction 'TTDCAtr' since it already exists.
Ignoring reaction 'AGPAT120' since it already exists.
Ignoring reaction 'DAGK120' since it already exists.
Ignoring reaction 'DASYN120' since it already exists.
Ignoring reaction 'ACOAAGPAT160' since it already exists.
Ignoring reaction 'AGPAT160' since it already exists.
Ignoring reaction 'DAGK160' since it already exists.
Ignoring reaction 'DASYN160' since it already exists.
Ignoring reaction 'LPLIPAL1G181' since it already exists.
Ignoring reaction 'PLIPA2G181' since it already exists.
Ignoring reaction 'ACTNDH' since it already exists.
Ignoring reaction 'ALAD_R' since it already exists.
Ignoring reaction 'ALCD1' since it already exists.
Ignoring reaction 'ALCD4' s

In [32]:
b_cel

Name,Bacteroides_cellulosilyticus_WH2
Memory address,7f2e181f3280
Number of metabolites,1702
Number of reactions,2779
Number of genes,968
Number of groups,77
Objective expression,1.0*biomass536 - 1.0*biomass536_reverse_50d4e
Compartments,"Cytoplasm, Extracellular"


In [33]:
write_sbml_model(b_cel, "../micom/models/Bacteria/Bacteroides_cellulosilyticus_WH2.xml")
validate_sbml_model("../micom/models/Bacteria/Bacteroides_cellulosilyticus_WH2.xml")

(<Model Bacteroides_cellulosilyticus_WH2 at 0x7f2dbda440d0>,
 {'SBML_FATAL': [],
  'SBML_ERROR': [],
  'SBML_SCHEMA_ERROR': [],
  'SBML_WARNING': [],
  'COBRA_FATAL': [],
  'COBRA_ERROR': [],
  'COBRA_WARNING': [],
  'COBRA_CHECK': []})